# Clasificación de supervivencia del Titanic

## Objetivo

Construir un flujo completo de clasificación con **scikit-learn** para predecir la supervivencia de un pasajero. Se comparan una **regresión logística** y un **árbol de decisión** mediante métricas, validación cruzada y visualizaciones. Finalmente, el modelo ganador se integra en una interfaz sencilla de **Gradio**.

> Este ejercicio es educativo. Describe patrones históricos del conjunto de datos y no debe utilizarse para tomar decisiones sobre personas.


## 1. Preparación del entorno

Este bloque instala Gradio e importa las librerías. `pandas` manipula los datos; `seaborn` y `matplotlib` generan gráficas; y `scikit-learn` se usa para preparar, entrenar y evaluar los modelos. La semilla global favorece resultados reproducibles.


In [ ]:
!pip -q install -U gradio

import io, os, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay,
                             classification_report)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
RANDOM_STATE = 42


## 2. Carga interactiva del archivo

En Google Colab se abrirá el selector para subir `Titanic.csv`. El bloque intenta varias codificaciones porque el archivo proporcionado utiliza caracteres de Windows (`cp1252`). Después normaliza el nombre de la columna de niños y valida que existan todas las variables necesarias.


In [ ]:
def leer_csv_robusto(contenido):
    for encoding in ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']:
        try:
            return pd.read_csv(io.BytesIO(contenido), encoding=encoding), encoding
        except UnicodeDecodeError:
            continue
    raise ValueError('No fue posible identificar la codificación del archivo.')

if os.path.exists('Titanic.csv'):
    with open('Titanic.csv', 'rb') as archivo:
        contenido = archivo.read()
    nombre_archivo = 'Titanic.csv'
else:
    from google.colab import files
    archivos_subidos = files.upload()
    nombre_archivo = next(iter(archivos_subidos))
    contenido = archivos_subidos[nombre_archivo]

df, encoding_usado = leer_csv_robusto(contenido)
df = df.rename(columns={'Ni¤os': 'Niños', 'Ninos': 'Niños'})

columnas_requeridas = ['Survived', 'Age', 'Passenger', 'Sex',
                       'Hermanos o Esposas', 'Niños', 'Tarifa']
faltantes = [c for c in columnas_requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')

print(f'Archivo: {nombre_archivo} | Codificación: {encoding_usado}')
print(f'Registros: {df.shape[0]} | Columnas: {df.shape[1]}')
display(df.head())


**Cómo interpretar la salida:** cada fila representa un pasajero. `Survived` es la variable objetivo. `Caso` es únicamente un identificador y se excluirá, porque un número consecutivo no debe utilizarse como característica predictiva.


## 3. Revisión de calidad de datos

Se inspeccionan tipos, valores faltantes, duplicados y categorías. Esta revisión evita entrenar con problemas ocultos. Aunque el archivo actual está completo, más adelante se incluyen imputadores para que el flujo tolere valores faltantes futuros.


In [ ]:
resumen_calidad = pd.DataFrame({
    'Tipo': df.dtypes.astype(str),
    'Valores faltantes': df.isna().sum(),
    'Porcentaje faltante': (df.isna().mean() * 100).round(2),
    'Valores únicos': df.nunique()
})
display(resumen_calidad)
print('Filas duplicadas:', df.duplicated().sum())
print('Distribución del objetivo:')
display(df['Survived'].value_counts().to_frame('Frecuencia'))


## 4. Análisis exploratorio de datos (EDA)

Las siguientes visualizaciones permiten conocer el balance del objetivo y explorar asociaciones entre supervivencia, sexo, clase, edad y tarifa. Son asociaciones descriptivas; no prueban causalidad.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='Survived', ax=axes[0,0])
axes[0,0].set_title('Balance de supervivencia')
axes[0,0].set_xlabel('Sobrevivió')

sns.countplot(data=df, x='Sex', hue='Survived', ax=axes[0,1])
axes[0,1].set_title('Supervivencia por sexo')

sns.countplot(data=df, x='Passenger', hue='Survived', ax=axes[1,0])
axes[1,0].set_title('Supervivencia por clase')

sns.histplot(data=df, x='Age', hue='Survived', kde=True,
             element='step', ax=axes[1,1])
axes[1,1].set_title('Distribución de edad por supervivencia')

plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Survived', y='Tarifa', ax=axes[0])
axes[0].set_title('Tarifa según supervivencia')

numericas_eda = ['Age', 'Hermanos o Esposas', 'Niños', 'Tarifa']
sns.heatmap(df[numericas_eda].corr(), annot=True, cmap='RdBu_r',
            center=0, fmt='.2f', ax=axes[1])
axes[1].set_title('Correlaciones entre variables numéricas')
plt.tight_layout(); plt.show()


In [ ]:
tasas = {
    'Tasa global': pd.crosstab(index=pd.Series(['Todos'] * len(df)),
                               columns=df['Survived'], normalize='index'),
    'Por sexo': pd.crosstab(df['Sex'], df['Survived'], normalize='index'),
    'Por clase': pd.crosstab(df['Passenger'], df['Survived'], normalize='index')
}
for titulo, tabla in tasas.items():
    print(f'\n{titulo} (%)')
    display((tabla * 100).round(1))


**Interpretación del EDA:** compare el porcentaje de `Yes` entre grupos. Una diferencia visible sugiere capacidad predictiva, pero no implica que la variable sea la causa de la supervivencia. Las distribuciones también permiten detectar asimetría y valores extremos, especialmente en `Tarifa`.


## 5. Preparación de variables y partición

`Survived` se convierte a 1/0. La división estratificada conserva aproximadamente la misma proporción de clases en entrenamiento y prueba. El conjunto de prueba permanece separado para estimar el desempeño con pasajeros no vistos.


In [ ]:
datos_modelo = df.copy()
datos_modelo['Survived'] = datos_modelo['Survived'].map({'Yes': 1, 'No': 0})
if datos_modelo['Survived'].isna().any():
    raise ValueError("Survived debe contener únicamente 'Yes' y 'No'.")

variables_numericas = ['Age', 'Hermanos o Esposas', 'Niños', 'Tarifa']
variables_categoricas = ['Passenger', 'Sex']
variables_predictoras = variables_numericas + variables_categoricas

X = datos_modelo[variables_predictoras]
y = datos_modelo['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)
print('Supervivencia entrenamiento:', round(y_train.mean(), 3))
print('Supervivencia prueba:', round(y_test.mean(), 3))


## 6. Pipelines de preprocesamiento y modelos

El `Pipeline` aplica exactamente las mismas transformaciones durante entrenamiento y predicción, reduciendo el riesgo de fuga de información. La regresión logística recibe variables escaladas; el árbol no necesita escalamiento. Las categorías se convierten con `OneHotEncoder` y se aceptan categorías desconocidas.


In [ ]:
prep_numerico_log = Pipeline([
    ('imputar', SimpleImputer(strategy='median')),
    ('escalar', StandardScaler())
])
prep_numerico_arbol = Pipeline([
    ('imputar', SimpleImputer(strategy='median'))
])
prep_categorico = Pipeline([
    ('imputar', SimpleImputer(strategy='most_frequent')),
    ('codificar', OneHotEncoder(handle_unknown='ignore'))
])

preprocesador_log = ColumnTransformer([
    ('num', prep_numerico_log, variables_numericas),
    ('cat', prep_categorico, variables_categoricas)
])
preprocesador_arbol = ColumnTransformer([
    ('num', prep_numerico_arbol, variables_numericas),
    ('cat', prep_categorico, variables_categoricas)
])

modelos = {
    'Regresión logística': Pipeline([
        ('preprocesador', preprocesador_log),
        ('modelo', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'Árbol de decisión': Pipeline([
        ('preprocesador', preprocesador_arbol),
        ('modelo', DecisionTreeClassifier(max_depth=4, min_samples_leaf=10,
                                          class_weight='balanced',
                                          random_state=RANDOM_STATE))
    ])
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(nombre, 'entrenado correctamente.')


## 7. Métricas en el conjunto de prueba

- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** de quienes fueron clasificados como sobrevivientes, cuántos realmente sobrevivieron.
- **Recall:** de quienes realmente sobrevivieron, cuántos detectó el modelo.
- **F1:** equilibrio entre precision y recall.
- **ROC-AUC:** capacidad de ordenar sobrevivientes por encima de no sobrevivientes a diferentes umbrales.


In [ ]:
resultados = []
predicciones = {}
probabilidades = {}

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    predicciones[nombre] = y_pred
    probabilidades[nombre] = y_prob
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

tabla_metricas = pd.DataFrame(resultados).set_index('Modelo').round(3)
display(tabla_metricas)
print('El valor más alto de cada columna representa el mejor resultado para esa métrica.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (nombre, y_pred) in zip(axes, predicciones.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['No', 'Sí'], cmap='Blues', ax=ax,
        colorbar=False
    )
    ax.set_title(f'Matriz de confusión\n{nombre}')
plt.tight_layout(); plt.show()


**Lectura de la matriz:** la diagonal contiene predicciones correctas. La celda inferior izquierda representa sobrevivientes que el modelo no detectó (falsos negativos); la superior derecha representa falsas alarmas (falsos positivos).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for nombre, y_prob in probabilidades.items():
    RocCurveDisplay.from_predictions(y_test, y_prob, name=nombre, ax=ax)
ax.plot([0, 1], [0, 1], '--', color='gray', label='Azar')
ax.set_title('Comparación de curvas ROC')
plt.show()

tabla_metricas.T.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), rot=0)
plt.title('Comparación de métricas en prueba')
plt.ylabel('Puntuación'); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()


## 8. Validación cruzada y selección del ganador

Una sola partición puede favorecer accidentalmente a un modelo. Por ello se aplica validación cruzada estratificada de cinco pliegues sobre el conjunto de entrenamiento. El ganador se elige por **ROC-AUC promedio**; si la diferencia es prácticamente nula, se utiliza F1 promedio como desempate.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'ROC-AUC': 'roc_auc', 'F1': 'f1', 'Accuracy': 'accuracy'}
filas_cv = []
for nombre, modelo in modelos.items():
    scores = cross_validate(modelo, X_train, y_train, cv=cv, scoring=scoring)
    filas_cv.append({
        'Modelo': nombre,
        'ROC-AUC CV': scores['test_ROC-AUC'].mean(),
        'Desv. ROC-AUC': scores['test_ROC-AUC'].std(),
        'F1 CV': scores['test_F1'].mean(),
        'Accuracy CV': scores['test_Accuracy'].mean()
    })

tabla_cv = pd.DataFrame(filas_cv).set_index('Modelo').round(3)
display(tabla_cv)

orden = tabla_cv.sort_values(['ROC-AUC CV', 'F1 CV'], ascending=False)
nombre_ganador = orden.index[0]
modelo_ganador = modelos[nombre_ganador]

# Reentrenamiento final con todos los datos disponibles para la aplicación.
modelo_ganador.fit(X, y)
display(Markdown(f'''### Modelo ganador: **{nombre_ganador}**

Se seleccionó por obtener el mayor ROC-AUC promedio en validación cruzada. Su ROC-AUC en prueba fue **{tabla_metricas.loc[nombre_ganador, 'ROC-AUC']:.3f}** y su F1 fue **{tabla_metricas.loc[nombre_ganador, 'F1']:.3f}**. Estas métricas describen desempeño predictivo en este dataset; no establecen relaciones causales.'''))


## 9. Interpretabilidad de ambos modelos

Los coeficientes positivos de la regresión aumentan la probabilidad estimada de supervivencia y los negativos la reducen, manteniendo las demás variables constantes. En el árbol, la importancia indica cuánto contribuyó cada variable a reducir impureza; no representa causalidad.


In [ ]:
def nombres_variables(pipeline):
    return pipeline.named_steps['preprocesador'].get_feature_names_out()

log_pipe = modelos['Regresión logística']
coef = pd.Series(log_pipe.named_steps['modelo'].coef_[0],
                 index=nombres_variables(log_pipe)).sort_values()
coef.tail(10).plot(kind='barh', figsize=(8,5), color='#2a9d8f')
plt.title('Coeficientes más positivos — regresión logística')
plt.xlabel('Coeficiente'); plt.tight_layout(); plt.show()

arbol_pipe = modelos['Árbol de decisión']
importancias = pd.Series(arbol_pipe.named_steps['modelo'].feature_importances_,
                         index=nombres_variables(arbol_pipe)).sort_values(ascending=False).head(10)
importancias.sort_values().plot(kind='barh', figsize=(8,5), color='#e9c46a')
plt.title('Importancia de variables — árbol de decisión')
plt.xlabel('Importancia'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(arbol_pipe.named_steps['modelo'],
          feature_names=nombres_variables(arbol_pipe),
          class_names=['No', 'Sí'], filled=True, rounded=True,
          proportion=True, fontsize=8)
plt.title('Estructura del árbol de decisión')
plt.show()


## 10. Interfaz de simulación con Gradio

La función crea una fila con los datos capturados, usa el pipeline ganador y devuelve la clase y su probabilidad. La interfaz no vuelve a entrenar el modelo: únicamente simula un nuevo pasajero con el modelo ya ajustado.


In [ ]:
import gradio as gr

def predecir_supervivencia(edad, clase, sexo, hermanos_esposas, ninos, tarifa):
    pasajero = pd.DataFrame([{
        'Age': edad,
        'Hermanos o Esposas': hermanos_esposas,
        'Niños': ninos,
        'Tarifa': tarifa,
        'Passenger': clase,
        'Sex': sexo
    }])
    probabilidad = float(modelo_ganador.predict_proba(pasajero)[0, 1])
    prediccion = 'Probable supervivencia' if probabilidad >= 0.50 else 'Probable no supervivencia'
    separador = chr(10) * 2
    return separador.join([
        f'### {prediccion}',
        f'**Probabilidad estimada de supervivencia:** {probabilidad:.1%}',
        f'**Modelo utilizado:** {nombre_ganador}',
        '> Simulación educativa basada en datos históricos; no representa certeza ni causalidad.'
    ])

with gr.Blocks(title='Simulador Titanic') as demo:
    gr.Markdown('# Simulador de supervivencia del Titanic')
    gr.Markdown('Modifique los datos del pasajero y presione **Predecir**.')
    with gr.Row():
        with gr.Column():
            edad = gr.Slider(0, 80, value=30, step=1, label='Edad')
            clase = gr.Dropdown(['First', 'Second', 'Third'], value='Third', label='Clase')
            sexo = gr.Radio(['Female', 'Male'], value='Male', label='Sexo')
        with gr.Column():
            hermanos = gr.Slider(0, 8, value=0, step=1, label='Hermanos o esposas')
            ninos = gr.Slider(0, 6, value=0, step=1, label='Niños')
            tarifa = gr.Slider(0, float(np.ceil(df['Tarifa'].max())), value=30,
                               step=1, label='Tarifa pagada')
    boton = gr.Button('Predecir', variant='primary')
    salida = gr.Markdown()
    boton.click(predecir_supervivencia,
                inputs=[edad, clase, sexo, hermanos, ninos, tarifa],
                outputs=salida)

demo.launch(share=True, debug=False)


## 11. Conclusiones y reflexión

1. El mejor modelo se determina mediante evidencia cuantitativa y validación cruzada, no por preferencia previa.
2. La regresión logística suele ofrecer mayor transparencia en la dirección de las asociaciones.
3. El árbol puede representar reglas no lineales, pero requiere limitar su complejidad para reducir sobreajuste.
4. Las métricas deben analizarse conjuntamente: accuracy por sí sola puede ocultar errores relevantes.
5. Los patrones del Titanic reflejan un contexto histórico específico y no son generalizables a decisiones actuales sobre personas.

**Pregunta de reflexión:** si el costo de no detectar a un sobreviviente potencial fuera mayor que el de generar una falsa alarma, ¿qué métrica y qué umbral de clasificación deberían priorizarse?
